# 🌊 ROMS Output Analysis: one year of the Indian Ocean

### This is notebook 3 of 3

| | Notebook | What you do |
|---|---|---|
| 1 | `Grid_IC_BC_Preparation_Steps.ipynb` | make grid, initial, boundary and forcing files |
| 2 | `ROMS_class_2026.ipynb` | compile and run ROMS for one year; save `roms_avg.nc`, `roms_his.nc`, `roms.log` to `MyDrive/ROMS_class` |
| **3** | **`ROMS_output_analysis.ipynb`** (this one) | **process and analyse the output** |

**What this notebook does**

| Part | Content |
|---|---|
| 1 | Load the output from Google Drive |
| 2 | Move u, v from the C-grid faces to ρ-points (cell centres) and rotate to east/north |
| 3 | Convert s-levels to real depths and interpolate to 35 standard depths (0.5 m – 6000 m) |
| (a) | **Interactive explorer**: maps and vertical sections; move the sliders to change date, depth, depth range and variable |
| (b) | Time series averaged over a region or at a point |
| (c) | Mixed-layer depth: mean, variability and seasonal cycle |
| (d) | West and East India Coastal Currents (WICC, EICC) and their seasonal reversal |
| (e) | Equatorial Kelvin waves and off-equatorial Rossby waves in Hovmöller diagrams |

Run the cells from top to bottom. Parts 2 and 3 take about a minute; everything after that is fast.

## 1. Load the model output from Google Drive

In notebook 2 the output was saved with:
```python
!mkdir -p /content/drive/MyDrive/ROMS_class && cp roms_*.nc roms.log /content/drive/MyDrive/ROMS_class/
```
| File | Content |
|---|---|
| `roms_avg.nc` | **5-day averages** of ζ, u, v, T, S, surface fluxes, KPP boundary-layer depth: used for most of the analysis |
| `roms_his.nc` | **10-day snapshots** (instantaneous fields) |

Both files also contain the grid (`h`, `mask_rho`, `pm`, `pn`, `angle`, `lon_rho`, `lat_rho`) and the s-coordinate parameters.

In [ ]:
%pip install -q cartopy gsw

In [ ]:
import os, shutil, warnings
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature
import ipywidgets as widgets
from IPython.display import display
warnings.filterwarnings('ignore', category=RuntimeWarning)       # all-NaN slices over land
plt.rcParams.update({'figure.dpi': 100})
PC = ccrs.PlateCarree()

DRIVE_DIR = '/content/drive/MyDrive/ROMS_class'     # where notebook 2 saved the output
WORK = '/content/roms_analysis'                       # local working copy (faster than reading from Drive)
os.makedirs(WORK, exist_ok=True)

if not os.path.isdir(DRIVE_DIR):
    from google.colab import drive
    drive.mount('/content/drive')
for f in ['roms_avg.nc', 'roms_his.nc', 'roms.log']:
    if not os.path.exists(f'{DRIVE_DIR}/{f}'):
        raise FileNotFoundError(f'{f} is not in {DRIVE_DIR}. Save the output at the end of notebook 2 first.')
    shutil.copy(f'{DRIVE_DIR}/{f}', WORK)
    print(f'copied {f:12s} {os.path.getsize(f"{WORK}/{f}") / 1e6:7.1f} MB')

In [ ]:
avg = xr.open_dataset(f'{WORK}/roms_avg.nc').load()
his = xr.open_dataset(f'{WORK}/roms_his.nc').load()

for name, ds in [('averages', avg), ('history', his)]:
    t = pd.to_datetime(ds.ocean_time.values)
    step = (t[1] - t[0]).days if len(t) > 1 else 0
    print(f'{name:9s}: {len(t):3d} records, every {step} days, {t[0]:%d %b %Y} → {t[-1]:%d %b %Y}')

# grid information (identical in both files)
mask = avg.mask_rho.where(avg.mask_rho == 1)          # 1 over ocean, NaN over land
LON, LAT = avg.lon_rho, avg.lat_rho
AREA = (1 / avg.pm / avg.pn).where(avg.mask_rho == 1) # cell area (m²), NaN over land
EXTENT = [float(LON.min()), float(LON.max()), float(LAT.min()), float(LAT.max())]
assert int(avg.Vtransform) == 2, 'this notebook assumes Vtransform = 2'
print(f'grid: {avg.sizes["xi_rho"]} x {avg.sizes["eta_rho"]} points, {avg.sizes["s_rho"]} s-levels, '
      f'lon {EXTENT[0]:.1f}–{EXTENT[1]:.1f}°E, lat {EXTENT[2]:.1f}–{EXTENT[3]:.1f}°N')

In [ ]:
def map_axes(ncols=1, nrows=1, figsize=None, extent=EXTENT):
    # cartopy map axes with land, coastline and labelled gridlines
    fig, axs = plt.subplots(nrows, ncols, figsize=figsize or (7.5 * ncols, 4.8 * nrows),
                            subplot_kw={'projection': PC}, squeeze=False)
    for ax in axs.flat:
        ax.set_extent(extent, crs=PC)
        ax.set_facecolor('#d9d4c7')                          # model land cells (NaN) shown as land
        ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#d9d4c7', zorder=2)
        ax.coastlines('50m', linewidth=0.6, zorder=3)
        gl = ax.gridlines(crs=PC, draw_labels=True, linewidth=0.4, color='gray', alpha=0.6, linestyle='--')
        gl.top_labels = gl.right_labels = False
    return fig, (axs[0, 0] if axs.size == 1 else axs.ravel())

def month_axis(ax, axis='x'):
    # one tick per month, labelled Jan, Feb, ...
    a = ax.xaxis if axis == 'x' else ax.yaxis
    a.set_major_locator(plt.matplotlib.dates.MonthLocator())
    a.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b'))

def nearest_ocean_point(lon0, lat0):
    # (j, i) of the ocean rho-point closest to (lon0, lat0)
    d2 = ((LON - lon0) * np.cos(np.deg2rad(lat0)))**2 + (LAT - lat0)**2
    j, i = np.unravel_index(int(np.nanargmin(d2.where(avg.mask_rho == 1).values)), d2.shape)
    return j, i

## 2. Velocities on ρ-points (7c)

ROMS uses an **Arakawa C-grid**: `u` sits on the west/east faces of each cell, `v` on the south/north faces, and T, S, ζ at the centre (ρ-point).
To plot or combine velocities with tracers, we

1. **average** the two `u` faces of each cell, and the two `v` faces, to the cell centre;
2. **rotate** by the grid angle so the components point **east** and **north** (here the angle is almost zero, but the step is needed for rotated grids).

```
      v(j+1)
   ┌────↑────┐
u(i) →  ρ   → u(i+1)        u_ρ = ½ [u(i) + u(i+1)]
   └────↑────┘              v_ρ = ½ [v(j) + v(j+1)]
      v(j)
```

In [ ]:
def uv_to_rho(ds):
    # C-grid u, v (all times and levels) -> eastward and northward velocity at rho-points
    U, V = ds.u.values, ds.v.values                    # (time, s, eta_rho, xi_u), (time, s, eta_v, xi_rho)
    ur = np.full(U.shape[:-1] + (U.shape[-1] + 1,), np.nan, dtype='f4')
    vr = np.full(V.shape[:-2] + (V.shape[-2] + 1, V.shape[-1]), np.nan, dtype='f4')
    ur[..., 1:-1] = 0.5 * (U[..., :-1] + U[..., 1:])   # average west + east face
    vr[..., 1:-1, :] = 0.5 * (V[..., :-1, :] + V[..., 1:, :])   # average south + north face
    a = ds.angle.values
    ue = ur * np.cos(a) - vr * np.sin(a)               # rotate grid -> east/north
    vn = vr * np.cos(a) + ur * np.sin(a)
    dims = ('ocean_time', 's_rho', 'eta_rho', 'xi_rho')
    coords = {'ocean_time': ds.ocean_time, 'lon_rho': LON, 'lat_rho': LAT}
    m = ds.mask_rho.values
    return (xr.DataArray(ue * m, dims=dims, coords=coords, name='u_east'),
            xr.DataArray(vn * m, dims=dims, coords=coords, name='v_north'))

u_avg, v_avg = uv_to_rho(avg)
u_his, v_his = uv_to_rho(his)
print('u at rho-points:', dict(u_avg.sizes))

In [ ]:
# Surface currents on one date (the top s-level is the surface layer)
t = min(30, avg.sizes['ocean_time'] - 1)                   # record number (~mid-May): try others
ue, vn = u_avg.isel(ocean_time=t, s_rho=-1), v_avg.isel(ocean_time=t, s_rho=-1)
spd = np.hypot(ue, vn)
fig, ax = map_axes(figsize=(11, 6))
p = ax.pcolormesh(LON, LAT, spd, transform=PC, cmap='viridis', vmin=0, vmax=1, shading='auto')
s = (slice(None, None, 2), slice(None, None, 2))
q = ax.quiver(LON.values[s], LAT.values[s], ue.values[s], vn.values[s], transform=PC, scale=10, width=0.0025, zorder=4)
ax.quiverkey(q, 0.88, -0.1, 0.5, '0.5 m/s', labelpos='E')
plt.colorbar(p, ax=ax, label='speed (m/s)', shrink=0.8)
ax.set_title(f'Surface currents, 5-day mean centred on {pd.Timestamp(avg.ocean_time.values[t]):%d %b %Y}');

## 3. From s-levels to real depths (7d)

ROMS levels follow the terrain, so level *k* is at a different depth in every column and at every time (the free surface moves).
With `Vtransform = 2` the depth of each ρ-point is

$$z = \zeta + (\zeta + h)\,\frac{h_c\,\sigma + h\,C(\sigma)}{h_c + h}$$

We then **interpolate linearly in the vertical** to fixed depths, so that maps "at 100 m" and sections compare like with like.
Points deeper than the local sea floor are left empty (NaN).

In [ ]:
DEPTHS = np.array([0.5, 5, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 110, 150, 200, 250, 300, 350, 400, 450, 500,
                   600, 700, 800, 900, 1000, 1250, 1500, 1750, 2000, 2500, 3000, 4000, 5000, 6000], dtype='f4')

def z_rho(ds):
    # depth (m, negative down) of every rho-point: (time, s, eta, xi)
    hc, h, zeta = float(ds.hc), ds.h, ds.zeta
    S = (hc * ds.s_rho + h * ds.Cs_r) / (hc + h)
    return (zeta + (zeta + h) * S).transpose('ocean_time', 's_rho', 'eta_rho', 'xi_rho')

def to_depths(fields, z, depths=DEPTHS):
    # Interpolate several (time, s, eta, xi) fields to fixed depths. Returns a dict of numpy arrays.
    Z = z.values.astype('f4')                          # s-levels increase upward: k = 0 is the bottom
    A = {k: v.values.astype('f4') for k, v in fields.items()}
    N = Z.shape[1]
    out = {k: np.full((Z.shape[0], len(depths)) + Z.shape[2:], np.nan, 'f4') for k in A}
    for n, d in enumerate(depths):
        zt = -d
        kb = (Z < zt).sum(axis=1) - 1                  # last level BELOW the target depth
        above_top = kb == N - 1                        # target shallower than the top level: use top value
        valid = kb >= 0                                # kb = -1: target deeper than the sea floor -> NaN
        k0 = np.clip(kb, 0, N - 2)[:, None]
        z0 = np.take_along_axis(Z, k0, 1)[:, 0]
        z1 = np.take_along_axis(Z, k0 + 1, 1)[:, 0]
        w = np.clip((zt - z0) / (z1 - z0), 0, 1)
        for k, a in A.items():
            a0 = np.take_along_axis(a, k0, 1)[:, 0]
            a1 = np.take_along_axis(a, k0 + 1, 1)[:, 0]
            val = np.where(above_top, a[:, -1], a0 + w * (a1 - a0))
            out[k][:, n] = np.where(valid, val, np.nan)
    return out

import gsw

def potential_density(dz):
    # potential density anomaly sigma_0 (kg/m3) from the model T (potential temperature) and S (practical salinity)
    p = gsw.p_from_z(-dz.depth.values[None, :, None, None], LAT.values[None, None])
    SA = gsw.SA_from_SP(dz.salt.values, p, LON.values[None, None], LAT.values[None, None])
    CT = gsw.CT_from_pt(SA, dz.temp.values)
    return xr.DataArray(gsw.sigma0(SA, CT).astype('f4'), dims=dz.temp.dims, coords=dz.temp.coords)

def mld_threshold(F, d, threshold, ref=10.0):
    # depth where |F - F(ref)| first exceeds the threshold, linearly interpolated between standard depths.
    # F: (time, depth, eta, xi) with F increasing downward (use -T for temperature)
    kref = int(np.argmin(np.abs(d - ref)))
    target = F[:, kref] + threshold
    T = -F; target = -target                               # same algorithm as for a decreasing temperature
    below = (T < target[:, None]) & (np.arange(len(d))[None, :, None, None] > kref)
    first = np.where(below.any(1), below.argmax(1), -1)    # first level colder than the target
    k1 = np.clip(first, 1, len(d) - 1)
    T0 = np.take_along_axis(T, (k1 - 1)[:, None], 1)[:, 0]; T1 = np.take_along_axis(T, k1[:, None], 1)[:, 0]
    w = np.clip((target - T0) / (T1 - T0), 0, 1)
    mld = d[k1 - 1] + w * (d[k1] - d[k1 - 1])
    bottom = np.nanmax(np.where(np.isfinite(T), d[None, :, None, None], np.nan), axis=1)
    mld = np.where(first == -1, bottom, mld)               # mixed to the bottom (shallow shelves)
    return mld * mask.values

def as_map(a, dz, name):
    return xr.DataArray(a, dims=('time', 'eta_rho', 'xi_rho'), coords={'time': dz.time, 'lon': LON, 'lat': LAT}, name=name)

def make_z_dataset(ds, ue, vn):
    z = z_rho(ds)
    m = ds.mask_rho.values
    res = to_depths({'temp': ds.temp * m, 'salt': ds.salt * m, 'u': ue, 'v': vn}, z)
    dims = ('time', 'depth', 'eta_rho', 'xi_rho')
    dz = xr.Dataset({k: (dims, v) for k, v in res.items()},
                    coords={'time': ds.ocean_time.values, 'depth': DEPTHS, 'lon': LON, 'lat': LAT})
    dz['speed'] = np.hypot(dz.u, dz.v)
    dz['zeta'] = (('time', 'eta_rho', 'xi_rho'), (ds.zeta * m).values)
    dz['sigma0'] = potential_density(dz)
    dz['mld_rho'] = as_map(mld_threshold(dz.sigma0.values, DEPTHS, 0.03), dz, 'mld_rho')   # see part (c)
    if 'Hsbl' in ds:
        dz['mld'] = (('time', 'eta_rho', 'xi_rho'), (-ds.Hsbl * m).values)    # KPP: stored as negative z
    for k, (ln, un) in {'temp': ('temperature', '°C'), 'salt': ('salinity', 'PSU'), 'u': ('eastward velocity', 'm/s'),
                        'v': ('northward velocity', 'm/s'), 'speed': ('current speed', 'm/s'),
                        'zeta': ('sea surface height', 'm'), 'mld': ('KPP boundary-layer depth', 'm'),
                        'sigma0': ('potential density anomaly', 'kg m-3'), 'mld_rho': ('mixed-layer depth (0.03 kg m-3)', 'm')}.items():
        if k in dz:
            dz[k].attrs = {'long_name': ln, 'units': un}
    return dz

%time zavg = make_z_dataset(avg, u_avg, v_avg)
zhis = make_z_dataset(his, u_his, v_his)
zavg

In [ ]:
# Check: a temperature profile on s-levels vs. the interpolated depths (Arabian Sea)
j, i = nearest_ocean_point(65, 15)
zz = z_rho(avg).isel(ocean_time=0, eta_rho=j, xi_rho=i)
fig, ax = plt.subplots(figsize=(4.5, 5))
ax.plot(avg.temp.isel(ocean_time=0, eta_rho=j, xi_rho=i), zz, 'o', ms=4, label='s-levels')
ax.plot(zavg.temp.isel(time=0, eta_rho=j, xi_rho=i), -zavg.depth, '-', lw=2, label='standard depths')
ax.set_ylim(-1500, 0); ax.set_xlabel('°C'); ax.set_ylabel('z (m)'); ax.legend()
ax.set_title(f'{float(LON[j, i]):.1f}°E, {float(LAT[j, i]):.1f}°N');

In [ ]:
# Save the processed data to Drive so you don't have to recompute it next time
zavg.to_netcdf(f'{DRIVE_DIR}/roms_avg_z.nc')
print('saved', f'{DRIVE_DIR}/roms_avg_z.nc')

## (a) Interactive explorer

Move the sliders: **date** changes the day, **depth** changes the level, **colour range** fixes the colour scale so you can compare dates.
Choose the variable from the list. `5-day averages` are smooth; `10-day snapshots` are instantaneous.

> In Colab the plot updates when you release a slider.

In [ ]:
VARS = {'Temperature (°C)': 'temp', 'Salinity (PSU)': 'salt', 'Sea surface height (m)': 'zeta',
        'Current speed (m/s)': 'speed', 'Eastward velocity u (m/s)': 'u', 'Northward velocity v (m/s)': 'v',
        'Mixed-layer depth, KPP (m)': 'mld', 'Mixed-layer depth, Δσ = 0.03 (m)': 'mld_rho'}
CMAPS = {'temp': 'RdYlBu_r', 'salt': 'viridis', 'zeta': 'RdBu_r', 'speed': 'viridis', 'u': 'RdBu_r', 'v': 'RdBu_r',
         'mld': 'cividis_r', 'mld_rho': 'cividis_r'}
DATA = {'5-day averages': zavg, '10-day snapshots': zhis}

def colour_range(dz, var, depth):
    a = dz[var] if dz[var].ndim == 3 else dz[var].sel(depth=depth, method='nearest')
    lo, hi = np.nanpercentile(a.values, [2, 98])
    if var in ('u', 'v', 'zeta'):                    # symmetric around zero
        lo, hi = -max(abs(lo), abs(hi)), max(abs(lo), abs(hi))
    return float(lo), float(hi)

w_data = widgets.Dropdown(options=list(DATA), description='Data')
w_var = widgets.Dropdown(options=list(VARS), description='Variable')
w_time = widgets.SelectionSlider(options=[(f'{pd.Timestamp(t):%d %b %Y}', n) for n, t in enumerate(zavg.time.values)],
                                 description='Date', continuous_update=False, layout=widgets.Layout(width='600px'))
w_depth = widgets.SelectionSlider(options=[(f'{d:g} m', float(d)) for d in DEPTHS], value=0.5,
                                  description='Depth', continuous_update=False, layout=widgets.Layout(width='600px'))
w_auto = widgets.Checkbox(value=True, description='auto colour range')
w_crange = widgets.FloatRangeSlider(value=(20, 30), min=-5, max=40, step=0.05, description='Colours',
                                    continuous_update=False, layout=widgets.Layout(width='600px'))
w_arrows = widgets.Checkbox(value=True, description='current arrows')

def _sync(*_):
    dz, var = DATA[w_data.value], VARS[w_var.value]
    w_time.options = [(f'{pd.Timestamp(t):%d %b %Y}', n) for n, t in enumerate(dz.time.values)]
    w_depth.disabled = dz[var].ndim == 3
    lo, hi = colour_range(dz, var, w_depth.value)
    span = hi - lo if hi > lo else 1
    w_crange.min, w_crange.max = -1e9, 1e9                # avoid min > max while updating
    w_crange.min, w_crange.max, w_crange.step = lo - span, hi + span, span / 200
    w_crange.value = (lo, hi)
w_data.observe(_sync, 'value'); w_var.observe(_sync, 'value')

def show_map(data, var_label, n, depth, auto, crange, arrows):
    dz, var = DATA[data], VARS[var_label]
    n = min(n, dz.sizes['time'] - 1)
    a = dz[var].isel(time=n)
    if a.ndim == 3:
        a = a.sel(depth=depth, method='nearest')
    lo, hi = colour_range(dz, var, depth) if auto else crange
    fig, ax = map_axes(figsize=(11, 6))
    p = ax.pcolormesh(LON, LAT, a, transform=PC, cmap=CMAPS[var], vmin=lo, vmax=hi, shading='auto')
    plt.colorbar(p, ax=ax, label=var_label, shrink=0.8)
    if arrows:
        u, v = dz.u.isel(time=n).sel(depth=depth, method='nearest'), dz.v.isel(time=n).sel(depth=depth, method='nearest')
        s = (slice(None, None, 2), slice(None, None, 2))
        ax.quiver(LON.values[s], LAT.values[s], u.values[s], v.values[s], transform=PC, scale=10, width=0.002, zorder=4)
    lev = '' if dz[var].ndim == 3 else f' at {depth:g} m'
    ax.set_title(f'{var_label}{lev} · {data} · {pd.Timestamp(dz.time.values[n]):%d %b %Y}')
    plt.show()

_sync()
out = widgets.interactive_output(show_map, {'data': w_data, 'var_label': w_var, 'n': w_time, 'depth': w_depth,
                                            'auto': w_auto, 'crange': w_crange, 'arrows': w_arrows})
display(widgets.VBox([widgets.HBox([w_data, w_var]), w_time, w_depth, widgets.HBox([w_auto, w_arrows]), w_crange, out]))

### Vertical sections

Choose a **latitude** (west–east section) or **longitude** (south–north section), the date, and the **depth range** to show.

In [ ]:
SECVARS = {k: v for k, v in VARS.items() if zavg[v].ndim == 4}     # 3-D variables only
s_var = widgets.Dropdown(options=list(SECVARS), description='Variable')
s_kind = widgets.ToggleButtons(options=['along latitude', 'along longitude'], description='Section')
s_pos = widgets.FloatSlider(value=10.0, min=EXTENT[2], max=EXTENT[3], step=0.5, description='Lat / lon',
                            continuous_update=False, layout=widgets.Layout(width='600px'))
s_time = widgets.SelectionSlider(options=[(f'{pd.Timestamp(t):%d %b %Y}', n) for n, t in enumerate(zavg.time.values)],
                                 description='Date', continuous_update=False, layout=widgets.Layout(width='600px'))
s_drange = widgets.IntRangeSlider(value=(0, 500), min=0, max=6000, step=50, description='Depth (m)',
                                  continuous_update=False, layout=widgets.Layout(width='600px'))

def _pos_limits(*_):
    if s_kind.value == 'along latitude':
        s_pos.min, s_pos.max, s_pos.value = EXTENT[2], EXTENT[3], 10.0
    else:
        s_pos.min, s_pos.max, s_pos.value = EXTENT[0], EXTENT[1], 65.0
s_kind.observe(_pos_limits, 'value')

def show_section(var_label, kind, pos, n, drange):
    var = SECVARS[var_label]
    a = zavg[var].isel(time=n)
    if kind == 'along latitude':
        j = int(np.argmin(np.abs(LAT.mean('xi_rho').values - pos)))
        sec, x, xl = a.isel(eta_rho=j), LON.isel(eta_rho=j), 'Longitude (°E)'
        hb, where = avg.h.isel(eta_rho=j), f'{float(LAT.isel(eta_rho=j).mean()):.1f}°N'
    else:
        i = int(np.argmin(np.abs(LON.mean('eta_rho').values - pos)))
        sec, x, xl = a.isel(xi_rho=i), LAT.isel(xi_rho=i), 'Latitude (°N)'
        hb, where = avg.h.isel(xi_rho=i), f'{float(LON.isel(xi_rho=i).mean()):.1f}°E'
    shown = sec.where((zavg.depth >= drange[0]) & (zavg.depth <= drange[1])).values
    lo, hi = np.nanpercentile(shown, [2, 98])               # colours fitted to the depth range shown
    if var in ('u', 'v'):
        lo, hi = -max(abs(lo), abs(hi)), max(abs(lo), abs(hi))
    fig, ax = plt.subplots(figsize=(12, 4.5))
    p = ax.pcolormesh(x, -zavg.depth, sec, cmap=CMAPS[var], vmin=lo, vmax=hi, shading='nearest')
    if var == 'temp':
        ax.contour(x, -zavg.depth, sec, levels=[20], colors='k', linewidths=1.2)   # 20 °C ≈ thermocline
    ax.fill_between(x, -hb.where(avg.mask_rho.isel(**({'eta_rho': j} if kind == 'along latitude' else {'xi_rho': i})) == 1, 0),
                    -6000, color='#d9d4c7')
    ax.set_ylim(-drange[1], -drange[0]); ax.set_xlabel(xl); ax.set_ylabel('Depth (m)')
    plt.colorbar(p, ax=ax, label=var_label)
    ax.set_title(f'{var_label} {kind} {where} · {pd.Timestamp(zavg.time.values[n]):%d %b %Y}')
    plt.show()

out2 = widgets.interactive_output(show_section, {'var_label': s_var, 'kind': s_kind, 'pos': s_pos, 'n': s_time,
                                                 'drange': s_drange})
display(widgets.VBox([widgets.HBox([s_var, s_kind]), s_pos, s_time, s_drange, out2]))

## (b) Time series for a region or a point

* **Region**: area-weighted average over all ocean cells inside a lon/lat box (cell area = 1/(pm·pn)).
* **Point**: the ocean grid cell nearest to a given lon/lat.

Edit the boxes below, or use the sliders in the interactive version.

In [ ]:
def series(lon_rng, lat_rng=None, depth=0.5, dz=zavg):
    # Region (lon_rng, lat_rng are (min, max)) or point (lon_rng, lat_rng are numbers): time series of all variables
    if np.ndim(lon_rng) == 0:
        j, i = nearest_ocean_point(lon_rng, lat_rng)
        pick = lambda a: a.isel(eta_rho=j, xi_rho=i)
        label = f'point {float(LON[j, i]):.1f}°E, {float(LAT[j, i]):.1f}°N'
    else:
        box = (LON >= lon_rng[0]) & (LON <= lon_rng[1]) & (LAT >= lat_rng[0]) & (LAT <= lat_rng[1])
        wgt = AREA.where(box)
        pick = lambda a: (a * wgt).sum(('eta_rho', 'xi_rho')) / wgt.where(np.isfinite(a)).sum(('eta_rho', 'xi_rho'))
        label = f'box {lon_rng[0]}–{lon_rng[1]}°E, {lat_rng[0]}–{lat_rng[1]}°N'
    df = pd.DataFrame({k: pick(dz[k] if dz[k].ndim == 3 else dz[k].sel(depth=depth, method='nearest')).values
                       for k in ['temp', 'salt', 'zeta', 'speed', 'u', 'v', 'mld']},
                      index=pd.to_datetime(dz.time.values))
    return df, label

def plot_series(df, label, depth, lon_rng, lat_rng):
    fig = plt.figure(figsize=(10, 7))
    axm = fig.add_subplot(3, 3, 1, projection=PC)
    axm.set_extent(EXTENT, crs=PC); axm.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#d9d4c7')
    axm.coastlines('50m', linewidth=0.5)
    if np.ndim(lon_rng) == 0:
        axm.plot(lon_rng, lat_rng, 'r*', ms=14, transform=PC)
    else:
        axm.plot([lon_rng[0], lon_rng[1], lon_rng[1], lon_rng[0], lon_rng[0]],
                 [lat_rng[0], lat_rng[0], lat_rng[1], lat_rng[1], lat_rng[0]], 'r-', lw=2, transform=PC)
    axm.set_title(label, fontsize=10)
    panels = [('temp', f'Temperature at {depth:g} m (°C)', 'C3'), ('salt', f'Salinity at {depth:g} m', 'C0'),
              ('zeta', 'Sea surface height (m)', 'C2'), ('speed', f'Current speed at {depth:g} m (m/s)', 'C4'),
              ('mld', 'Mixed-layer depth, KPP (m)', 'C1')]
    for n, (k, title, c) in enumerate(panels):
        ax = fig.add_subplot(3, 3, n + 2) if n < 2 else fig.add_subplot(3, 3, (n + 2, n + 2))
        ax.plot(df.index, df[k], '-o', ms=3, color=c)
        if k == 'mld':
            ax.invert_yaxis()
        ax.set_title(title, fontsize=11); ax.grid(alpha=0.3); month_axis(ax)
    ax = fig.add_subplot(3, 3, (7, 9))
    ax.plot(df.index, df.u, label='u (east)'); ax.plot(df.index, df.v, label='v (north)'); ax.axhline(0, color='k', lw=0.6)
    ax.set_title(f'Velocity components at {depth:g} m (m/s)', fontsize=11); ax.legend(); ax.grid(alpha=0.3); month_axis(ax)
    plt.tight_layout(); plt.show()

# Example 1: central Arabian Sea box
# lon_rng, lat_rng, depth = (60, 70), (10, 20), 0.5
# df, label = series(lon_rng, lat_rng, depth)
# plot_series(df, label, depth, lon_rng, lat_rng)

In [ ]:
# # Example 2: a point in the southern Bay of Bengal, at 50 m
# pt_lon, pt_lat, depth = 88.0, 8.0, 50
# df, label = series(pt_lon, pt_lat, depth)
# plot_series(df, label, depth, pt_lon, pt_lat)
# df.describe().round(3)

In [ ]:
# Interactive version: choose a box (or tick 'point' to use the box centre) and a depth
b_lon = widgets.FloatRangeSlider(value=(80, 95), min=EXTENT[0], max=EXTENT[1], step=1, description='Lon',
                                 continuous_update=False, layout=widgets.Layout(width='600px'))
b_lat = widgets.FloatRangeSlider(value=(5, 20), min=EXTENT[2], max=EXTENT[3], step=1, description='Lat',
                                 continuous_update=False, layout=widgets.Layout(width='600px'))
b_point = widgets.Checkbox(value=False, description='point (box centre)')
b_depth = widgets.SelectionSlider(options=[(f'{d:g} m', float(d)) for d in DEPTHS[:21]], value=0.5,
                                  description='Depth', continuous_update=False, layout=widgets.Layout(width='600px'))

def _series_gui(lon, lat, point, depth):
    if point:
        lon, lat = (lon[0] + lon[1]) / 2, (lat[0] + lat[1]) / 2
    plot_series(*series(lon, lat, depth), depth, lon, lat)

display(widgets.VBox([b_lon, b_lat, b_point, b_depth,
                      widgets.interactive_output(_series_gui, {'lon': b_lon, 'lat': b_lat, 'point': b_point, 'depth': b_depth})]))

## (c) Mixed-layer depth (MLD)

Three estimates:

* **KPP boundary-layer depth** (`Hsbl`), written by the model's mixing scheme (the depth of active turbulent mixing);
* **Density criterion** (the standard one): the depth where potential density first exceeds its 10 m value by **0.03 kg m⁻³** (de Boyer Montégut et al., 2004);
* **Temperature criterion**: the depth where temperature first falls **0.2 °C below its 10 m value** (the *isothermal layer*).

In most of the ocean the density and temperature MLDs agree. Where fresh water sits on top (the **Bay of Bengal**: river runoff and monsoon rain), salinity sets the density, so the mixed layer is shallower than the isothermal layer.
The difference is the **barrier layer**: it separates the mixed layer from the cold thermocline water and keeps the surface warm.

We look at the **annual mean**, its **variability** (standard deviation over the year), and the **seasonal cycle** in a region or at a point.

In [ ]:
# mld_rho (density criterion) was already computed in Part 3; add the temperature criterion and the barrier layer
zavg['mld_T'] = as_map(mld_threshold(-zavg.temp.values, DEPTHS, 0.2), zavg, 'mld_T')
zavg['barrier'] = zavg.mld_T - zavg.mld_rho

maps = [(zavg.mld_rho.mean('time'), 'Annual mean MLD, Δσ = 0.03 kg m⁻³ (m)', 'cividis_r', 0, 100),
        (zavg.mld_rho.std('time'), 'Variability: std of the MLD over the year (m)', 'magma_r', 0, 40),
        (zavg.mld.mean('time'), 'Annual mean KPP boundary layer (m)', 'cividis_r', 0, 100),
        (zavg.barrier.mean('time'), 'Annual mean barrier layer = MLD(T) − MLD(ρ) (m)', 'Blues', 0, 40)]
fig, ax = map_axes(2, 2, figsize=(10, 5))
for a, (fld, title, cm, lo, hi) in zip(ax, maps):
    p = a.pcolormesh(LON, LAT, fld, transform=PC, cmap=cm, vmin=lo, vmax=hi, shading='auto')
    plt.colorbar(p, ax=a, shrink=0.75); a.set_title(title)
plt.tight_layout()

In [ ]:
def mld_series(lon_rng, lat_rng):
    if np.ndim(lon_rng) == 0:
        j, i = nearest_ocean_point(lon_rng, lat_rng)
        out = {k: zavg[k].isel(eta_rho=j, xi_rho=i).values for k in ['mld_rho', 'mld_T', 'mld']}
        spread = None
        label = f'point {float(LON[j, i]):.1f}°E, {float(LAT[j, i]):.1f}°N'
    else:
        box = (LON >= lon_rng[0]) & (LON <= lon_rng[1]) & (LAT >= lat_rng[0]) & (LAT <= lat_rng[1])
        wgt = AREA.where(box)
        out = {k: ((zavg[k] * wgt).sum(('eta_rho', 'xi_rho')) / wgt.where(np.isfinite(zavg[k])).sum(('eta_rho', 'xi_rho'))).values
               for k in ['mld_rho', 'mld_T', 'mld']}
        spread = zavg.mld_rho.where(box).std(('eta_rho', 'xi_rho')).values   # spatial spread inside the box
        label = f'box {lon_rng[0]}–{lon_rng[1]}°E, {lat_rng[0]}–{lat_rng[1]}°N'
    df = pd.DataFrame(out, index=pd.to_datetime(zavg.time.values))
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.5), gridspec_kw={'width_ratios': [2.2, 1]})
    ax[0].plot(df.index, df.mld_rho, '-o', ms=3, color='C0', lw=2, label='Δσ = 0.03 kg m⁻³')
    ax[0].plot(df.index, df.mld_T, '-', color='C1', label='ΔT = 0.2 °C (isothermal layer)')
    ax[0].plot(df.index, df.mld, '--', color='C2', label='KPP boundary layer')
    if spread is not None:
        ax[0].fill_between(df.index, df.mld_rho - spread, df.mld_rho + spread, alpha=0.2, color='C0', label='±1 std in box')
    ax[0].invert_yaxis(); ax[0].set_ylabel('MLD (m)'); ax[0].legend(); ax[0].grid(alpha=0.3); month_axis(ax[0])
    ax[0].set_title(f'Mixed-layer depth · {label}')
    clim = df[['mld_rho', 'mld_T']].groupby(df.index.month).mean().rename(
        columns={'mld_rho': 'Δσ = 0.03', 'mld_T': 'ΔT = 0.2 °C'})
    clim.index = [pd.Timestamp(2000, m, 1).strftime('%b') for m in clim.index]
    clim.plot(kind='bar', ax=ax[1], rot=0, color=['C0', 'C1']); ax[1].invert_yaxis(); ax[1].set_title('Monthly mean'); ax[1].set_ylabel('MLD (m)')
    plt.tight_layout(); plt.show()
    print(f'MLD (density): mean {df.mld_rho.mean():.1f} m, std {df.mld_rho.std():.1f} m, '
          f'range {df.mld_rho.min():.1f}–{df.mld_rho.max():.1f} m · isothermal layer {df.mld_T.mean():.1f} m · '
          f'KPP {df.mld.mean():.1f} m · barrier layer {(df.mld_T - df.mld_rho).mean():.1f} m')
    return df

# _ = mld_series((60, 70), (10, 20))        # Arabian Sea: deep in winter (NE monsoon) and summer (SW monsoon)

In [ ]:
_ = mld_series((85, 92), (10, 20))        # Bay of Bengal: shallow mixed layer above a thick barrier layer

In [ ]:
_ = mld_series(65.0, 15.0)                # a single point

## (d) West and East India Coastal Currents

The coastal currents around India **reverse with the monsoon**:

| | Summer monsoon (Jun–Sep) | Winter monsoon (Nov–Feb) | Spring (Mar–Apr) |
|---|---|---|---|
| **WICC** (west coast) | equatorward (south) | poleward (north) | |
| **EICC** (east coast) | | equatorward (south) | poleward (north) |

**Method**
1. For each grid row between the latitudes below, find the **first ocean cells next to the Indian coast** (walking out from inland India).
2. Take the **alongshore** velocity, positive **poleward** (along the coast towards higher latitude), averaged over the top 50 m.
3. Average along the coast → one number per date, and also keep the full latitude–time picture (Hovmöller).

> At ~100 km resolution the coastal currents are only one or two grid cells wide, so expect weaker and broader currents than in observations.

In [ ]:
def coastal_cells(side, lat_min, lat_max, start_lon=78.0, width=2):
    # Ocean cells along the Indian coast. side='west': walk west from inland India; 'east': walk east.
    m = avg.mask_rho.values
    cells = []
    for j in range(avg.sizes['eta_rho']):
        lat_row = float(LAT[j].mean())
        if not (lat_min <= lat_row <= lat_max):
            continue
        i0 = int(np.argmin(np.abs(LON[j].values - start_lon)))
        if m[j, i0] == 1:                     # start point must be on land (inside India)
            continue
        step = -1 if side == 'west' else 1
        i = i0
        while 0 <= i + step < m.shape[1] and m[j, i] == 0:
            i += step
        if m[j, i] == 1:
            cells.append([(j, i + step * k) for k in range(width) if 0 <= i + step * k < m.shape[1] and m[j, i + step * k] == 1])
    return cells

def alongshore(cells, top=50, dz=zavg):
    # depth-mean (0-top m) alongshore velocity, positive poleward; returns DataFrame (time x coast latitude)
    lay = dz.sel(depth=dz.depth[dz.depth <= top])
    u = lay.u.mean('depth').values; v = lay.v.mean('depth').values
    pts = np.array([[float(LON[c[0]]), float(LAT[c[0]])] for c in cells])
    # local coast direction from neighbouring coastal points, pointing to higher latitude
    tx = np.gradient(pts[:, 0]) * np.cos(np.deg2rad(pts[:, 1])); ty = np.gradient(pts[:, 1])
    tx, ty = np.convolve(tx, np.ones(3) / 3, 'same'), np.convolve(ty, np.ones(3) / 3, 'same')
    norm = np.hypot(tx, ty); tx, ty = tx / norm, ty / norm
    vals = np.stack([np.nanmean([u[:, j, i] * tx[n] + v[:, j, i] * ty[n] for j, i in c], axis=0)
                     for n, c in enumerate(cells)], axis=1)
    return pd.DataFrame(vals, index=pd.to_datetime(dz.time.values), columns=np.round(pts[:, 1], 1))

west = coastal_cells('west', 8.5, 21.5)
east = coastal_cells('east', 10.5, 20.5)
fig, ax = map_axes(figsize=(8, 6), extent=[65, 95, 3, 25])
for cells, c, name in [(west, 'C3', 'WICC'), (east, 'C0', 'EICC')]:
    for row in cells:
        for j, i in row:
            ax.plot(float(LON[j, i]), float(LAT[j, i]), 's', color=c, ms=7, transform=PC, zorder=5)
    ax.plot([], [], 's', color=c, label=f'{name} section cells')
ax.legend(loc='lower left'); ax.set_title('Coastal grid cells used for the WICC and EICC');

In [ ]:
wicc, eicc = alongshore(west), alongshore(east)
fig, ax = plt.subplots(figsize=(15, 5))
for df, c, name in [(wicc, 'C3', 'WICC (west coast)'), (eicc, 'C0', 'EICC (east coast)')]:
    s = df.mean(axis=1)
    ax.plot(s.index, s * 100, '-o', ms=4, color=c, lw=2, label=name)
ax.axhline(0, color='k', lw=0.8)
for m1, m2, txt in [(6, 9, 'SW monsoon'), (11, 12, 'NE monsoon')]:
    y = wicc.index[0].year
    ax.axvspan(pd.Timestamp(y, m1, 1), pd.Timestamp(y, m2, 30), color='0.9', zorder=0)
    ax.text(pd.Timestamp(y, m1, 5), ax.get_ylim()[1] * 0.9, txt, fontsize=10, color='0.35')
ax.set_ylabel('alongshore velocity, 0–50 m (cm/s)\n+ poleward   − equatorward')
ax.set_title('Seasonal reversal of the India coastal currents'); ax.legend(); ax.grid(alpha=0.3)
month_axis(ax);

In [ ]:
# # Latitude–time picture along each coast (Hovmöller): red = poleward, blue = equatorward
# fig, ax = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
# for a, (df, name) in zip(ax, [(wicc, 'WICC: west coast'), (eicc, 'EICC: east coast')]):
#     lim = np.nanpercentile(np.abs(df.values), 95) * 100
#     p = a.pcolormesh(df.columns, df.index, df.values * 100, cmap='RdBu_r', vmin=-lim, vmax=lim, shading='nearest')
#     a.set_xlabel('Latitude along the coast (°N)'); a.set_title(name)
#     plt.colorbar(p, ax=a, label='cm/s, + poleward')
# month_axis(ax[0], 'y')
# plt.tight_layout()

## (e) Kelvin and Rossby waves in Hovmöller diagrams

A **Hovmöller diagram** plots a quantity against **longitude (x)** and **time (y)**. A wave shows up as a tilted stripe; the slope is its phase speed.
We use the **sea surface height anomaly** (each point minus its annual mean), which tracks the movement of the thermocline.

| Wave | Where | Direction | Theory: speed of the first baroclinic mode (c ≈ 2.5 m/s) |
|---|---|---|---|
| Equatorial **Kelvin** | 1°S–1°N | **eastward** | c ≈ 2.5 m/s ≈ 1.9° longitude per day |
| **Rossby** at 13°N | Bay of Bengal, 78–100°E | **westward** | c_R = β c² / f² ≈ 0.13 m/s ≈ 0.12°/day |
| **Rossby** at 6°N | Arabian Sea, 50–78°E | **westward** | c_R ≈ 0.6 m/s ≈ 0.5°/day |

The dashed lines show these theoretical speeds as a guide; compare their slope with the stripes.

In [ ]:
ssha = zavg.zeta - zavg.zeta.mean('time')                     # sea-level anomaly (m)

def hovmoller(lat_min, lat_max, lon_min, lon_max, title, speed_ms=None, direction=+1, ax=None, highpass_days=None):
    band = (LAT >= lat_min) & (LAT <= lat_max) & (LON >= lon_min) & (LON <= lon_max)
    h = ssha.where(band).mean('eta_rho')                      # average across the latitude band
    lon = LON.where(band).mean('eta_rho')
    keep = np.isfinite(lon.values) & np.isfinite(h.values).any(0)
    h, lon = h.isel(xi_rho=np.where(keep)[0]), lon.values[keep]
    if highpass_days:                                         # remove the slow seasonal signal
        nrec = max(3, int(round(highpass_days / 5)))
        h = h - h.rolling(time=nrec, center=True, min_periods=1).mean()
    t = pd.to_datetime(zavg.time.values)
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    lim = float(np.nanpercentile(np.abs(h.values), 98)) * 100
    p = ax.contourf(lon, t, h.values * 100, levels=np.linspace(-lim, lim, 21), cmap='RdBu_r', extend='both')
    plt.colorbar(p, ax=ax, label='SSH anomaly (cm)')
    if speed_ms:                                              # guide lines with the theoretical phase speed
        deg_per_day = speed_ms * 86400 / (111e3 * np.cos(np.deg2rad((lat_min + lat_max) / 2)))
        span_days = (lon.max() - lon.min()) / deg_per_day
        for t0 in pd.date_range(t[0], t[-1], freq='60D'):
            x0, x1 = (lon.min(), lon.max()) if direction > 0 else (lon.max(), lon.min())
            ax.plot([x0, x1], [t0, t0 + pd.Timedelta(days=span_days)], 'k--', lw=1)
        ax.set_ylim(t[0], t[-1])
    ax.set_xlabel('Longitude (°E)'); ax.set_title(title)
    month_axis(ax, 'y')
    return ax

# fig, ax = plt.subplots(1, 3, figsize=(21, 8), sharey=True)
# hovmoller(-1, 1, 40, 100, 'Equatorial Kelvin waves (1°S–1°N)\ndashed: 2.5 m/s eastward', 2.5, +1, ax[0])
# hovmoller(12.5, 13.5, 78, 100, 'Rossby waves, Bay of Bengal (13°N)\ndashed: 0.13 m/s westward', 0.13, -1, ax[1])
# hovmoller(5.5, 6.5, 50, 78, 'Rossby waves, Arabian Sea (6°N)\ndashed: 0.6 m/s westward', 0.6, -1, ax[2])
# plt.tight_layout()

In [ ]:
# Kelvin waves are fast (≈1 month to cross the basin) and sit on top of a large seasonal signal.
# Removing a 60-day running mean (a high-pass filter) makes the eastward-tilted pulses stand out.
# Look for the pulses in April–May and October–December: they are forced by the westerly winds of the
# monsoon transitions (Wyrtki jets).
# fig, ax = plt.subplots(1, 2, figsize=(15, 8), sharey=True)
# hovmoller(-1, 1, 40, 100, 'Equatorial SSH anomaly (seasonal signal included)', 2.5, +1, ax[0])
# hovmoller(-1, 1, 40, 100, 'Equatorial SSH, 60-day running mean removed\ndashed: Kelvin wave, 2.5 m/s eastward', 2.5, +1, ax[1],
#           highpass_days=60)
# plt.tight_layout()

In [ ]:
# Try your own band: latitude range, longitude range, title, guide speed (m/s), direction (+1 east, -1 west)
ax = hovmoller(-5, -4, 50, 100, 'Your choice: 4–5°S', 0.8, -1)

### Questions to discuss

1. When do the equatorial Kelvin waves appear? Relate them to the spring and autumn **Wyrtki jets** driven by westerly winds during the monsoon transitions.
2. What happens to a Kelvin wave when it reaches the eastern boundary (Sumatra)? Look for coastally trapped waves travelling into the Bay of Bengal and Rossby waves radiating west.
3. Is the Rossby wave faster at 6°N or 13°N? Why does the speed depend on latitude (hint: c_R = β c² / f²)?
4. Do the WICC and EICC reverse when expected? How could a 1° model, with closed boundaries and monthly winds, bias them?

### References
* de Boyer Montégut et al. (2004), *J. Geophys. Res.* 109, C12003: mixed-layer depth criteria
* Shankar, Vinayachandran & Unnikrishnan (2002), *Prog. Oceanogr.* 52, 63–120: the monsoon currents in the north Indian Ocean
* Schott & McCreary (2001), *Prog. Oceanogr.* 51, 1–123: the monsoon circulation of the Indian Ocean
* Wyrtki (1973), *Science* 181, 262–264: equatorial jets in the Indian Ocean
* Chelton et al. (1998), *J. Phys. Oceanogr.* 28, 433–460: Rossby radius and wave speeds